## Model Training (LightGBM Regression)

This notebook trains a LightGBM regressor to predict the **3-month-ahead SST anomaly** (`target_ssta`).
It uses the same feature CSVs as the classification notebook; only the target and objective change.

### Imports and Setup

Import libraries for data loading, LightGBM regression, and evaluation.
If the kernel starts inside `notebooks/`, move up one level so relative paths to the data files work.

In [121]:
import os
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import (
    classification_report,
    f1_score,
    mean_absolute_error,
    accuracy_score,
    root_mean_squared_error,
)

if Path.cwd().name == "notebooks":
    os.chdir("..")
print("Working directory:", Path.cwd())

Working directory: /Users/abiralshrestha/Documents/Python/Break through tech/AWS-1A-climate-risk-modeling-for-supply-chain-resiliency


### Data Loading

Load the train, validation, and test feature CSVs created in the feature engineering step.

In [122]:
train = pd.read_csv("data/processed/features-train.csv")
val = pd.read_csv("data/processed/features-val.csv")
test = pd.read_csv("data/processed/features-test.csv")

print("Train:", train.shape, "| Val:", val.shape, "| Test:", test.shape)

Train: (1988, 43) | Val: (1529, 43) | Test: (694, 43)


### Feature and Target Separation

In [123]:
drop_cols = [
    "site_id",
    "ym",
    "year",
    "month",
    "current_risk",
    "target_ssta",
    "target_risk",
]

X_train = train.drop(columns=drop_cols)
y_train = train["target_ssta"]

X_val = val.drop(columns=drop_cols)
y_val = val["target_ssta"]

X_test = test.drop(columns=drop_cols)
y_test = test["target_ssta"]

print("Features:", list(X_train.columns))
print("n_features:", X_train.shape[1])

Features: ['ssta', 'zonal_wind_anom', 'mer_wind_anom', 'air_temp_anom', 'humidity', 'lat', 'lon', 'month_sin', 'month_cos', 'ssta_change1', 'ssta_change3', 'ssta_lag1', 'ssta_lag2', 'ssta_lag3', 'ssta_lag6', 'zonal_wind_anom_lag1', 'zonal_wind_anom_lag2', 'zonal_wind_anom_lag3', 'zonal_wind_anom_lag6', 'mer_wind_anom_lag1', 'mer_wind_anom_lag2', 'mer_wind_anom_lag3', 'mer_wind_anom_lag6', 'air_temp_anom_lag1', 'air_temp_anom_lag2', 'air_temp_anom_lag3', 'air_temp_anom_lag6', 'ssta_roll3', 'ssta_roll6', 'zonal_wind_anom_roll3', 'zonal_wind_anom_roll6', 'nino34_ssta_roll3', 'basin_ssta', 'basin_zwind_anom', 'nino34_ssta', 'west_pac_zwind_anom']
n_features: 36


### Model Training

Train a LightGBM regressor with early stopping on the validation RMSE.

In [141]:
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

lgbm_params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.04,
    "num_leaves": 8,
    "max_depth": 7,
    "min_child_samples": 25,
    "lambda_l1": 0.2,
    "lambda_l2": 1.5,
    "verbosity": -1,
}

model = lgb.train(
    params=lgbm_params,
    train_set=train_data,
    num_boost_round=250,
    valid_sets=[train_data, val_data],
    valid_names=["train", "val"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=30),
        lgb.log_evaluation(period=5),
    ],
)

print("Best iteration:", model.best_iteration)

Training until validation scores don't improve for 30 rounds
[5]	train's rmse: 0.727988	val's rmse: 0.830301
[10]	train's rmse: 0.672054	val's rmse: 0.819967
[15]	train's rmse: 0.628815	val's rmse: 0.809357
[20]	train's rmse: 0.594474	val's rmse: 0.797321
[25]	train's rmse: 0.568364	val's rmse: 0.788329
[30]	train's rmse: 0.547817	val's rmse: 0.780567
[35]	train's rmse: 0.530742	val's rmse: 0.776956
[40]	train's rmse: 0.516165	val's rmse: 0.775254
[45]	train's rmse: 0.504219	val's rmse: 0.769391
[50]	train's rmse: 0.492306	val's rmse: 0.766046
[55]	train's rmse: 0.483017	val's rmse: 0.762609
[60]	train's rmse: 0.474503	val's rmse: 0.762055
[65]	train's rmse: 0.466503	val's rmse: 0.761341
[70]	train's rmse: 0.458902	val's rmse: 0.761666
[75]	train's rmse: 0.452687	val's rmse: 0.761015
[80]	train's rmse: 0.446928	val's rmse: 0.76035
[85]	train's rmse: 0.442041	val's rmse: 0.760476
[90]	train's rmse: 0.43716	val's rmse: 0.759909
[95]	train's rmse: 0.43284	val's rmse: 0.759882
[100]	train'

### Prediction

Generate continuous anomaly predictions on the validation set.

In [142]:
y_pred_val = model.predict(X_val, num_iteration=model.best_iteration)

pred_summary = pd.DataFrame({"y_true": y_val, "y_pred": y_pred_val})
pred_summary.describe().round(3)

,y_true,y_pred
count,1529.000,1529.000
mean,-0.244,-0.038
std,0.824,0.351
min,-3.450,-1.167
25%,-0.760,-0.253
50%,-0.152,-0.029
75%,0.321,0.200
max,2.808,1.481


### Train vs Validation Metrics

Compare train and validation scores to check for overfitting.

In [143]:
def regression_report(y_true, y_pred, split_name):
    rmse = root_mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    print(f"{split_name}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE:  {mae:.4f}")
    return {"split": split_name, "rmse": rmse, "mae": mae}
    
y_pred_train = model.predict(X_train, num_iteration=model.best_iteration)

train_metrics = regression_report(y_train, y_pred_train, "Training")
val_metrics = regression_report(y_val, y_pred_val, "Validation")

Training
  RMSE: 0.4213
  MAE:  0.3010
Validation
  RMSE: 0.7591
  MAE:  0.5626


### Classify SSTA predictions

Map continuous SSTA predictions to NOAA risk tiers for comparison with baselines.


In [144]:
def classify_risk(anomaly):
    if pd.isna(anomaly):
        return np.nan
    if anomaly >= 1.5:
        return "Extreme Warm"
    if anomaly >= 0.5:
        return "Moderate Warm"
    if anomaly <= -1.5:
        return "Extreme Cool"
    if anomaly <= -0.5:
        return "Moderate Cool"
    return "Neutral"

### Validation Dummy Baseline

Always predict the training majority class (`Neutral`).


In [145]:
val_dummy_preds = ["Neutral"] * len(val)
val_dummy_f1 = f1_score(val["target_risk"], val_dummy_preds, average="macro", zero_division=0)
val_dummy_acc = accuracy_score(val["target_risk"], val_dummy_preds)

print("=== Validation Dummy Baseline ===")
print(classification_report(val["target_risk"], val_dummy_preds, zero_division=0))
print(f"Accuracy: {val_dummy_acc:.4f} | Macro F1: {val_dummy_f1:.4f}")

=== Validation Dummy Baseline ===
               precision    recall  f1-score   support

 Extreme Cool       0.00      0.00      0.00       110
 Extreme Warm       0.00      0.00      0.00        17
Moderate Cool       0.00      0.00      0.00       408
Moderate Warm       0.00      0.00      0.00       237
      Neutral       0.50      1.00      0.66       757

     accuracy                           0.50      1529
    macro avg       0.10      0.20      0.13      1529
 weighted avg       0.25      0.50      0.33      1529

Accuracy: 0.4951 | Macro F1: 0.1325


### Validation Persistence Baseline

Assume risk in 3 months equals today's tier (`target_risk = current_risk`).


In [146]:
val_persist_preds = val["current_risk"]
val_persist_f1 = f1_score(val["target_risk"], val_persist_preds, average="macro", zero_division=0)
val_persist_acc = accuracy_score(val["target_risk"], val_persist_preds)

print("=== Validation Persistence Baseline ===")
print(classification_report(val["target_risk"], val_persist_preds, zero_division=0))
print(f"Accuracy: {val_persist_acc:.4f} | Macro F1: {val_persist_f1:.4f}")

=== Validation Persistence Baseline ===
               precision    recall  f1-score   support

 Extreme Cool       0.29      0.26      0.28       110
 Extreme Warm       0.00      0.00      0.00        17
Moderate Cool       0.47      0.45      0.46       408
Moderate Warm       0.38      0.39      0.38       237
      Neutral       0.64      0.66      0.65       757

     accuracy                           0.52      1529
    macro avg       0.36      0.35      0.35      1529
 weighted avg       0.52      0.52      0.52      1529

Accuracy: 0.5245 | Macro F1: 0.3533


### Validation LightGBM vs baselines

Bin continuous LightGBM predictions and compare Macro F1 / accuracy to Dummy and Persistence.


In [147]:
val_model_risk = pd.Series(y_pred_val, index=val.index).map(classify_risk)
val_model_f1 = f1_score(val["target_risk"], val_model_risk, average="macro", zero_division=0)
val_model_acc = accuracy_score(val["target_risk"], val_model_risk)

print("=== Validation LightGBM ===")
print(classification_report(val["target_risk"], val_model_risk, zero_division=0))
print(f"Accuracy: {val_model_acc:.4f} | Macro F1: {val_model_f1:.4f}")

val_comparison = pd.DataFrame([
    {"model": "Dummy", "accuracy": val_dummy_acc, "macro_f1": val_dummy_f1},
    {"model": "Persistence", "accuracy": val_persist_acc, "macro_f1": val_persist_f1},
    {"model": "LightGBM", "accuracy": val_model_acc, "macro_f1": val_model_f1},
])
val_comparison

=== Validation LightGBM ===
               precision    recall  f1-score   support

 Extreme Cool       0.00      0.00      0.00       110
 Extreme Warm       0.00      0.00      0.00        17
Moderate Cool       0.52      0.18      0.26       408
Moderate Warm       0.27      0.07      0.11       237
      Neutral       0.52      0.92      0.67       757

     accuracy                           0.51      1529
    macro avg       0.26      0.23      0.21      1529
 weighted avg       0.44      0.51      0.42      1529

Accuracy: 0.5141 | Macro F1: 0.2091


,model,accuracy,macro_f1
0,Dummy,0.495095,0.132458
1,Persistence,0.524526,0.353263
2,LightGBM,0.514061,0.209110


### Test Dummy Baseline

Same Dummy rule as validation: always predict `Neutral`.


In [148]:
test_dummy_preds = ["Neutral"] * len(test)
test_dummy_f1 = f1_score(test["target_risk"], test_dummy_preds, average="macro", zero_division=0)
test_dummy_acc = accuracy_score(test["target_risk"], test_dummy_preds)

print("=== Test Dummy Baseline ===")
print(classification_report(test["target_risk"], test_dummy_preds, zero_division=0))
print(f"Accuracy: {test_dummy_acc:.4f} | Macro F1: {test_dummy_f1:.4f}")

=== Test Dummy Baseline ===
               precision    recall  f1-score   support

 Extreme Cool       0.00      0.00      0.00         7
 Extreme Warm       0.00      0.00      0.00       169
Moderate Cool       0.00      0.00      0.00        89
Moderate Warm       0.00      0.00      0.00       117
      Neutral       0.45      1.00      0.62       312

     accuracy                           0.45       694
    macro avg       0.09      0.20      0.12       694
 weighted avg       0.20      0.45      0.28       694

Accuracy: 0.4496 | Macro F1: 0.1241


### Test Persistence Baseline

Same Persistence rule as validation: `target_risk = current_risk`.


In [149]:
test_persist_preds = test["current_risk"]
test_persist_f1 = f1_score(test["target_risk"], test_persist_preds, average="macro", zero_division=0)
test_persist_acc = accuracy_score(test["target_risk"], test_persist_preds)

print("=== Test Persistence Baseline ===")
print(classification_report(test["target_risk"], test_persist_preds, zero_division=0))
print(f"Accuracy: {test_persist_acc:.4f} | Macro F1: {test_persist_f1:.4f}")

=== Test Persistence Baseline ===
               precision    recall  f1-score   support

 Extreme Cool       0.00      0.00      0.00         7
 Extreme Warm       0.66      0.63      0.64       169
Moderate Cool       0.13      0.17      0.15        89
Moderate Warm       0.21      0.23      0.22       117
      Neutral       0.57      0.50      0.53       312

     accuracy                           0.44       694
    macro avg       0.31      0.31      0.31       694
 weighted avg       0.47      0.44      0.45       694

Accuracy: 0.4380 | Macro F1: 0.3077


### Test LightGBM vs baselines

Score continuous test predictions, then bin and compare to Dummy and Persistence.


In [150]:
y_pred_test = model.predict(X_test, num_iteration=model.best_iteration)
test_metrics = regression_report(y_test, y_pred_test, "Test")

test_model_risk = pd.Series(y_pred_test, index=test.index).map(classify_risk)
test_model_f1 = f1_score(test["target_risk"], test_model_risk, average="macro", zero_division=0)
test_model_acc = accuracy_score(test["target_risk"], test_model_risk)

print("\n=== Test LightGBM ===")
print(classification_report(test["target_risk"], test_model_risk, zero_division=0))
print(f"Accuracy: {test_model_acc:.4f} | Macro F1: {test_model_f1:.4f}")

test_comparison = pd.DataFrame([
    {"model": "Dummy", "accuracy": test_dummy_acc, "macro_f1": test_dummy_f1},
    {"model": "Persistence", "accuracy": test_persist_acc, "macro_f1": test_persist_f1},
    {"model": "LightGBM (binned)", "accuracy": test_model_acc, "macro_f1": test_model_f1},
])
test_comparison

Test
  RMSE: 1.1653
  MAE:  0.8262

=== Test LightGBM ===
               precision    recall  f1-score   support

 Extreme Cool       0.00      0.00      0.00         7
 Extreme Warm       0.77      0.34      0.47       169
Moderate Cool       0.43      0.07      0.12        89
Moderate Warm       0.20      0.32      0.25       117
      Neutral       0.58      0.78      0.66       312

     accuracy                           0.50       694
    macro avg       0.40      0.30      0.30       694
 weighted avg       0.54      0.50      0.47       694

Accuracy: 0.4957 | Macro F1: 0.2993


,model,accuracy,macro_f1
0,Dummy,0.449568,0.124056
1,Persistence,0.438040,0.307712
2,LightGBM (binned),0.495677,0.299268


## Feature Set Experiments (LightGBM)

Compare LightGBM regressors trained on different feature subsets.
Predictions are binned to risk classes for Macro F1 / Accuracy, and compared to Dummy and Persistence.

Uses the **36 features** already in `features-*.csv` (includes `west_pac_zwind_anom`).


### Feature lists

- **FEATURES_FULL (36):** all model features  
- **FEATURES_NOISE (13):** humidity, month sin/cos, historical local wind lags/rolls  
- **FEATURES_TRIMMED (23):** FULL without NOISE (includes `west_pac_zwind_anom`)  
- **FEATURES_PHYSICAL (16):** lean thermodynamic / basin set (no lat/lon, fewer lags)


In [151]:
FEATURES_NOISE = [
    "humidity", "month_sin", "month_cos",
    "zonal_wind_anom_lag1", "zonal_wind_anom_lag2", "zonal_wind_anom_lag3", "zonal_wind_anom_lag6",
    "zonal_wind_anom_roll3", "zonal_wind_anom_roll6",
    "mer_wind_anom_lag1", "mer_wind_anom_lag2", "mer_wind_anom_lag3", "mer_wind_anom_lag6",
]

FEATURES_TRIMMED = [
    "ssta", "ssta_lag1", "ssta_lag2", "ssta_lag3", "ssta_lag6",
    "ssta_roll3", "ssta_roll6", "ssta_change1", "ssta_change3",
    "basin_ssta", "nino34_ssta", "nino34_ssta_roll3", "basin_zwind_anom", "west_pac_zwind_anom",
    "air_temp_anom", "air_temp_anom_lag1", "air_temp_anom_lag2", "air_temp_anom_lag3", "air_temp_anom_lag6",
    "lat", "lon",
    "zonal_wind_anom", "mer_wind_anom",
]

FEATURES_FULL = FEATURES_TRIMMED + FEATURES_NOISE

FEATURES_PHYSICAL = [
    "ssta", "ssta_lag1", "ssta_lag2", "ssta_lag3", "ssta_roll3", "ssta_change1", "ssta_change3",
    "basin_ssta", "nino34_ssta", "nino34_ssta_roll3", "basin_zwind_anom", "west_pac_zwind_anom",
    "air_temp_anom", "air_temp_anom_lag1", "zonal_wind_anom", "mer_wind_anom",
]

print(f"Full Feature Set:     {len(FEATURES_FULL)} features")
print(f"Trimmed Feature Set:  {len(FEATURES_TRIMMED)} features")
print(f"Noise Feature Set:    {len(FEATURES_NOISE)} features")
print(f"Physical Feature Set: {len(FEATURES_PHYSICAL)} features")

Full Feature Set:     36 features
Trimmed Feature Set:  23 features
Noise Feature Set:    13 features
Physical Feature Set: 16 features


### `record_result()` helper

Logs Val Macro F1 / Accuracy / Extreme F1, plus RMSE / MAE when continuous preds are provided.
Also flags whether Macro F1 and RMSE beat the Persistence baselines.


In [152]:
results = []
persist_f1 = val_persist_f1  # risk-tier persistence
persist_rmse = root_mean_squared_error(y_val, val["ssta"])  # continuous persistence
persist_mae = mean_absolute_error(y_val, val["ssta"])


def record_result(name, num_feats, y_pred_risk, y_pred_ssta=None):
    y_pred_risk = pd.Series(y_pred_risk).reset_index(drop=True)
    y_true_risk = val["target_risk"].reset_index(drop=True)

    f1 = f1_score(y_true_risk, y_pred_risk, average="macro", zero_division=0)
    acc = accuracy_score(y_true_risk, y_pred_risk)
    ec_f1 = f1_score(y_true_risk == "Extreme Cool", y_pred_risk == "Extreme Cool", zero_division=0)
    ew_f1 = f1_score(y_true_risk == "Extreme Warm", y_pred_risk == "Extreme Warm", zero_division=0)

    if y_pred_ssta is None:
        rmse = mae = None
        beats_persist_rmse = None
    else:
        y_pred_ssta = pd.Series(y_pred_ssta).reset_index(drop=True)
        rmse = root_mean_squared_error(y_val.reset_index(drop=True), y_pred_ssta)
        mae = mean_absolute_error(y_val.reset_index(drop=True), y_pred_ssta)
        beats_persist_rmse = rmse < persist_rmse

    entry = {
        "Model / Strategy": name,
        "Features": num_feats,
        "Macro F1": round(f1, 4),
        "Accuracy": round(acc, 4),
        "RMSE": None if rmse is None else round(rmse, 4),
        "MAE": None if mae is None else round(mae, 4),
        "Extreme Cool F1": round(ec_f1, 3),
        "Extreme Warm F1": round(ew_f1, 3),
        "Beats Persist F1?": f1 > persist_f1,
        "Beats Persist RMSE?": beats_persist_rmse,
    }
    results.append(entry)

    rmse_txt = "n/a" if rmse is None else f"{rmse:.4f}"
    mae_txt = "n/a" if mae is None else f"{mae:.4f}"
    print(
        f"{name:<42} | F1: {f1:.4f} | Acc: {acc:.4f} | "
        f"RMSE: {rmse_txt} | MAE: {mae_txt} | "
        f"Beat F1/RMSE: {f1 > persist_f1}/{beats_persist_rmse}"
    )
    return f1

### Train LightGBM on each feature set

Same hyperparameters as the main model. Original `model` is not overwritten.
Continuous preds are binned with `classify_risk` before scoring.


In [153]:
def run_lgbm_experiment(name, features):
    train_set = lgb.Dataset(X_train[features], label=y_train)
    val_set = lgb.Dataset(X_val[features], label=y_val, reference=train_set)

    exp_model = lgb.train(
        params=lgbm_params,
        train_set=train_set,
        num_boost_round=100,
        valid_sets=[val_set],
        valid_names=["val"],
        callbacks=[
            lgb.early_stopping(stopping_rounds=20),
            lgb.log_evaluation(period=0),
        ],
    )

    y_hat = exp_model.predict(X_val[features], num_iteration=exp_model.best_iteration)
    y_hat_risk = pd.Series(y_hat).map(classify_risk)
    record_result(name, len(features), y_hat_risk, y_pred_ssta=y_hat)
    return exp_model


print(f"Continuous persistence baseline — Val RMSE: {persist_rmse:.4f} | MAE: {persist_mae:.4f}")
print()

# Baselines: Dummy has no continuous SSTA preds; risk Persistence pairs with ssta persistence for RMSE
record_result("Dummy (always Neutral)", 0, val_dummy_preds)
record_result(
    "Persistence (current_risk / ssta)",
    0,
    val_persist_preds,
    y_pred_ssta=val["ssta"],
)

# Feature-set experiments
LightGBM_Full = run_lgbm_experiment("LightGBM Full (36)", FEATURES_FULL)
LightGBM_Trimmed = run_lgbm_experiment("LightGBM Trimmed (23)", FEATURES_TRIMMED)
LightGBM_Physical = run_lgbm_experiment("LightGBM Physical (16)", FEATURES_PHYSICAL)

Continuous persistence baseline — Val RMSE: 0.7817 | MAE: 0.5779

Dummy (always Neutral)                     | F1: 0.1325 | Acc: 0.4951 | RMSE: n/a | MAE: n/a | Beat F1/RMSE: False/None
Persistence (current_risk / ssta)          | F1: 0.3533 | Acc: 0.5245 | RMSE: 0.7817 | MAE: 0.5779 | Beat F1/RMSE: False/False
Training until validation scores don't improve for 20 rounds
Did not meet early stopping. Best iteration is:
[99]	val's rmse: 0.759127
LightGBM Full (36)                         | F1: 0.2088 | Acc: 0.5154 | RMSE: 0.7591 | MAE: 0.5624 | Beat F1/RMSE: False/True
Training until validation scores don't improve for 20 rounds
Did not meet early stopping. Best iteration is:
[100]	val's rmse: 0.738804
LightGBM Trimmed (23)                      | F1: 0.2256 | Acc: 0.5199 | RMSE: 0.7388 | MAE: 0.5464 | Beat F1/RMSE: False/True
Training until validation scores don't improve for 20 rounds
Did not meet early stopping. Best iteration is:
[96]	val's rmse: 0.737711
LightGBM Physical (16)       

### Results table


In [154]:
results_df = pd.DataFrame(results)
results_df

,Model / Strategy,Features,Macro F1,Accuracy,RMSE,MAE,Extreme Cool F1,Extreme Warm F1,Beats Persist F1?,Beats Persist RMSE?
0,Dummy (always Neutral),0,0.1325,0.4951,NaN,NaN,0.000,0.0,False,None
1,Persistence (current_risk / ssta),0,0.3533,0.5245,0.7817,0.5779,0.276,0.0,False,False
2,LightGBM Full (36),36,0.2088,0.5154,0.7591,0.5624,0.000,0.0,False,True
3,LightGBM Trimmed (23),23,0.2256,0.5199,0.7388,0.5464,0.000,0.0,False,True
4,LightGBM Physical (16),16,0.2330,0.5167,0.7377,0.5462,0.034,0.0,False,True


## Experiment: Residual (Delta) Regression

Hypothesis: absolute `target_ssta` regression shrinks toward Neutral. Predicting the **3-month change**
`delta = target_ssta - ssta`, then reconstructing `pred_ssta = ssta + pred_delta`, should start from
Persistence and only deviate when features support a real shift.

Uses the same features and `lgbm_params` as the main model. Does not overwrite `model`.


### Train delta model

Target label is `target_ssta - ssta`. Early stopping monitors delta RMSE on validation.


In [156]:
y_train_delta = train["target_ssta"] - train["ssta"]
y_val_delta = val["target_ssta"] - val["ssta"]
y_test_delta = test["target_ssta"] - test["ssta"]

train_delta_set = lgb.Dataset(X_train, label=y_train_delta)
val_delta_set = lgb.Dataset(X_val, label=y_val_delta, reference=train_delta_set)

model_delta = lgb.train(
    params=lgbm_params,
    train_set=train_delta_set,
    num_boost_round=250,
    valid_sets=[train_delta_set, val_delta_set],
    valid_names=["train", "val"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=30),
        lgb.log_evaluation(period=5),
    ],
)

print("Best iteration (delta):", model_delta.best_iteration)


Training until validation scores don't improve for 30 rounds
[5]	train's rmse: 0.643177	val's rmse: 0.761328
[10]	train's rmse: 0.615589	val's rmse: 0.748537
[15]	train's rmse: 0.591416	val's rmse: 0.739126
[20]	train's rmse: 0.570942	val's rmse: 0.732097
[25]	train's rmse: 0.554941	val's rmse: 0.728513
[30]	train's rmse: 0.541281	val's rmse: 0.724853
[35]	train's rmse: 0.528511	val's rmse: 0.722392
[40]	train's rmse: 0.518203	val's rmse: 0.721525
[45]	train's rmse: 0.508372	val's rmse: 0.722277
[50]	train's rmse: 0.499381	val's rmse: 0.722732
[55]	train's rmse: 0.491298	val's rmse: 0.722037
[60]	train's rmse: 0.483696	val's rmse: 0.721802
[65]	train's rmse: 0.477023	val's rmse: 0.721421
[70]	train's rmse: 0.471219	val's rmse: 0.721114
[75]	train's rmse: 0.465328	val's rmse: 0.722618
[80]	train's rmse: 0.460117	val's rmse: 0.723211
[85]	train's rmse: 0.454642	val's rmse: 0.724478
[90]	train's rmse: 0.449703	val's rmse: 0.725273
[95]	train's rmse: 0.44487	val's rmse: 0.726615
Early stop

### Reconstruct absolute SSTA and score

Convert delta predictions back to absolute anomaly space, then report RMSE/MAE and risk-tier Macro F1.


In [157]:
y_pred_val_delta = model_delta.predict(X_val, num_iteration=model_delta.best_iteration)
y_pred_test_delta = model_delta.predict(X_test, num_iteration=model_delta.best_iteration)

y_pred_val_from_delta = val["ssta"].to_numpy() + y_pred_val_delta
y_pred_test_from_delta = test["ssta"].to_numpy() + y_pred_test_delta

print("=== Reconstructed SSTA (delta model) ===")
delta_val_reg = regression_report(y_val, y_pred_val_from_delta, "Validation")
delta_test_reg = regression_report(y_test, y_pred_test_from_delta, "Test")

print(
    f"\nPredicted SSTA range (val): {y_pred_val_from_delta.min():.2f} to {y_pred_val_from_delta.max():.2f}"
)
print(f"True SSTA range (val):      {y_val.min():.2f} to {y_val.max():.2f}")


=== Reconstructed SSTA (delta model) ===
Validation
  RMSE: 0.7208
  MAE:  0.5281
Test
  RMSE: 0.9667
  MAE:  0.7031

Predicted SSTA range (val): -2.22 to 2.23
True SSTA range (val):      -3.45 to 2.81


### Risk-tier comparison vs baselines

Bin reconstructed predictions with `classify_risk` and compare to Dummy, Persistence, and absolute LightGBM.


In [158]:
val_delta_risk = pd.Series(y_pred_val_from_delta, index=val.index).map(classify_risk)
test_delta_risk = pd.Series(y_pred_test_from_delta, index=test.index).map(classify_risk)

val_delta_f1 = f1_score(val["target_risk"], val_delta_risk, average="macro", zero_division=0)
val_delta_acc = accuracy_score(val["target_risk"], val_delta_risk)
test_delta_f1 = f1_score(test["target_risk"], test_delta_risk, average="macro", zero_division=0)
test_delta_acc = accuracy_score(test["target_risk"], test_delta_risk)

print("=== Validation — LightGBM delta ===")
print(classification_report(val["target_risk"], val_delta_risk, zero_division=0))

val_delta_compare = pd.DataFrame(
    [
        {"model": "Dummy", "accuracy": val_dummy_acc, "macro_f1": val_dummy_f1},
        {"model": "Persistence", "accuracy": val_persist_acc, "macro_f1": val_persist_f1},
        {"model": "LightGBM (absolute)", "accuracy": val_model_acc, "macro_f1": val_model_f1},
        {"model": "LightGBM (delta)", "accuracy": val_delta_acc, "macro_f1": val_delta_f1},
    ]
).round(3)
print(val_delta_compare)

print("=== Test — LightGBM delta ===")
print(classification_report(test["target_risk"], test_delta_risk, zero_division=0))

test_delta_compare = pd.DataFrame(
    [
        {"model": "Dummy", "accuracy": test_dummy_acc, "macro_f1": test_dummy_f1},
        {"model": "Persistence", "accuracy": test_persist_acc, "macro_f1": test_persist_f1},
        {"model": "LightGBM (absolute)", "accuracy": test_model_acc, "macro_f1": test_model_f1},
        {"model": "LightGBM (delta)", "accuracy": test_delta_acc, "macro_f1": test_delta_f1},
    ]
).round(3)
print(test_delta_compare)

print(
    f"Beats Persistence on Val Macro F1?  {val_delta_f1 > val_persist_f1} "
    f"({val_delta_f1:.3f} vs {val_persist_f1:.3f})"
)
print(
    f"Beats Persistence on Test Macro F1? {test_delta_f1 > test_persist_f1} "
    f"({test_delta_f1:.3f} vs {test_persist_f1:.3f})"
)


=== Validation — LightGBM delta ===
               precision    recall  f1-score   support

 Extreme Cool       0.29      0.08      0.13       110
 Extreme Warm       0.00      0.00      0.00        17
Moderate Cool       0.51      0.38      0.44       408
Moderate Warm       0.34      0.17      0.23       237
      Neutral       0.59      0.83      0.69       757

     accuracy                           0.54      1529
    macro avg       0.35      0.29      0.30      1529
 weighted avg       0.50      0.54      0.50      1529

                 model  accuracy  macro_f1
0                Dummy     0.495     0.132
1          Persistence     0.525     0.353
2  LightGBM (absolute)     0.514     0.209
3     LightGBM (delta)     0.545     0.296
=== Test — LightGBM delta ===
               precision    recall  f1-score   support

 Extreme Cool       0.00      0.00      0.00         7
 Extreme Warm       0.84      0.58      0.69       169
Moderate Cool       0.28      0.13      0.18        89
